# Step-by-Step One-Norm Calculation

This notebook walks through the seven steps of the Hamiltonian one-norm ($\lambda$) computation using `OneNormCalculator`. Each step is run individually so you can inspect intermediate results.

We use the 2x2x2 LCBO test data for metallic hydrogen.

> **Prerequisite:** The data file `data/lcbo_2x2x2.h5` must exist. It is generated by running notebook **02_gpaw_extraction** (or provided by a collaborator).

In [1]:
import warnings
import numpy as np
from numpy.exceptions import ComplexWarning
from bloch_paw import PawReader, OneNormCalculator

# Suppress expected numerical warnings from zero C-tensor eigenvalues
# (hydrogen has negligible PAW corrections, so division-by-zero in the
# Frobenius-norm rescaling is harmless — the result is correctly zero)
warnings.filterwarnings("ignore", category=RuntimeWarning, module="numpy")
warnings.filterwarnings("ignore", category=ComplexWarning)

DATA_FILE = "../data/lcbo_2x2x2.h5"

## Setup: Load Data and Create Calculator

We use `PawReader.to_calculator_inputs()` to get a dictionary of arrays, then pass it directly to `OneNormCalculator`.

In [2]:
reader = PawReader(DATA_FILE)
inputs = reader.to_calculator_inputs()

calc = OneNormCalculator(**inputs, thr_rank=3e-5, sv_floor=1e-12, scale_floor=1e-12)

print(f"Nk = {calc.Nk}  (k-points)")
print(f"Nb = {calc.Nb}  (bands per k-point)")
print(f"Npw = {calc.Npw} (plane waves, G ≠ 0)")
print(f"Grid = {calc.Nx} × {calc.Ny} × {calc.Nz} = {calc.N_grid} points")
print(f"Supercell volume V = {calc.V:.4f} ų")

Nk = 8  (k-points)
Nb = 5  (bands per k-point)
Npw = 511 (plane waves, G ≠ 0)
Grid = 8 × 8 × 8 = 512 points
Supercell volume V = 12.3577 ų


## Step 1: Soft Modes --- `compute_soft_modes()`

This step FFTs the pair density $\tilde{\rho}(\mathbf{k}, \mathbf{k} \oplus \mathbf{Q})$ from real space to reciprocal space, then performs a batched SVD over band indices to extract the singular-value spectrum.

For each $(\mathbf{Q}, \mathbf{G}, \mathbf{k})$ block, the SVD of the $N_b \times N_b$ matrix yields singular values $\sigma$, which are split into $\pm\sigma/2$ pairs to form a $2 N_b$ spectrum. This convention follows the Chebyshev amplitude amplification scheme.

The result is a dictionary `{J: f}` where `f` has shape $(N_k, N_\text{pw}, N_k, 2 N_b)$. The index $J \in \{0, 1\}$ labels the spin channel (identical for spin-free systems).

In [3]:
f_soft = calc.compute_soft_modes()

print(f"Soft eigenvalue keys: {list(f_soft.keys())}")
print(f"Shape of f_soft[0]: {f_soft[0].shape}")
print(f"  = (Nk_Q={f_soft[0].shape[0]}, Npw={f_soft[0].shape[1]}, "
      f"Nk={f_soft[0].shape[2]}, 2·Nb={f_soft[0].shape[3]})")
print(f"\nSpectrum statistics (J=0):")
print(f"  max |f|:  {np.max(np.abs(f_soft[0])):.6f}")
print(f"  Non-zero entries: {np.count_nonzero(f_soft[0])} / {f_soft[0].size} "
      f"({np.count_nonzero(f_soft[0]) / f_soft[0].size:.1%})")

Soft eigenvalue keys: [0, 1]
Shape of f_soft[0]: (8, 511, 8, 10)
  = (Nk_Q=8, Npw=511, Nk=8, 2·Nb=10)

Spectrum statistics (J=0):
  max |f|:  0.013844
  Non-zero entries: 236130 / 327040 (72.2%)


## Step 2: $C$-Tensor Eigendecomposition --- `diagonalize_Ca()`

Reshapes each $C^a$ tensor from $(n_a, n_a, n_a, n_a)$ into a symmetric matrix $M$ of shape $(n_a^2, n_a^2)$, then diagonalises: $M = O \cdot \text{diag}(\varepsilon) \cdot O^T$.

The eigenvalues $\varepsilon^a_{rs}$ weight the PAW contribution to $\lambda$. Their magnitudes determine how important each partial-wave pair is in the LCU decomposition.

> **Note on the test data:** For metallic hydrogen, the PAW on-site corrections are negligibly small ($C^a \approx 0$). This means Steps 2, 3, and 5 will produce near-zero results. For heavier elements (e.g., iron), the PAW terms contribute significantly to $\lambda$.

In [4]:
ca_eigs = calc.diagonalize_Ca()

for a, eigs in sorted(ca_eigs.items()):
    eps = eigs["eps"]
    O = eigs["O"]
    print(f"Atom {a}:")
    print(f"  Eigenvalues ε^a: shape {eps.shape}")
    print(f"  Eigenvectors O:  shape {O.shape}")
    print(f"  Eigenvalue range: [{eps.min():.4f}, {eps.max():.4f}]")
    print(f"  Number of significant eigenvalues (|ε| > 1e-5): "
          f"{np.sum(np.abs(eps) > 1e-5)}")
    print(f"  Largest |ε| values: {np.sort(np.abs(eps))[::-1][:5]}")

Atom 0:
  Eigenvalues ε^a: shape (25,)
  Eigenvectors O:  shape (25, 25)
  Eigenvalue range: [0.0000, 0.0000]
  Number of significant eigenvalues (|ε| > 1e-5): 0
  Largest |ε| values: [0. 0. 0. 0. 0.]


## Step 3: PAW Modes --- `compute_paw_modes()`

This step contracts $D^a$ with the $C$-tensor eigenvectors $O$ from Step 2, then performs a batched SVD over band indices. The result captures how the PAW augmentation contribution is distributed over $(\mathbf{Q}, \mathbf{k})$ blocks.

Returns a tuple of `(f_paw, eps_sign)`:
- `f_paw`: dictionary `{(atom, J): array}` with shape $(N_k, P, N_k, 2 N_b)$ where $P = n_a^2$
- `eps_sign`: dictionary `{atom: sign(ε)}` tracking eigenvalue signs for the block encoding

In [5]:
f_paw, eps_sign = calc.compute_paw_modes()

print("PAW mode tensors:")
for (a, J), fp in sorted(f_paw.items()):
    print(f"  (atom={a}, J={J}): shape {fp.shape}, "
          f"non-zero: {np.count_nonzero(fp)}/{fp.size} "
          f"({np.count_nonzero(fp)/fp.size:.1%})")

print("\nEigenvalue signs:")
for a, signs in sorted(eps_sign.items()):
    n_pos = np.sum(signs > 0)
    n_neg = np.sum(signs < 0)
    n_zero = np.sum(signs == 0)
    print(f"  Atom {a}: {n_pos} positive, {n_neg} negative, {n_zero} zero")

PAW mode tensors:
  (atom=0, J=0): shape (8, 25, 8, 10), non-zero: 0/16000 (0.0%)
  (atom=0, J=1): shape (8, 25, 8, 10), non-zero: 0/16000 (0.0%)

Eigenvalue signs:
  Atom 0: 0 positive, 0 negative, 25 zero


## Step 4: Soft Contribution $\xi$ --- `compute_xi()`

Accumulates the soft (plane-wave) contribution to the one-norm:

$$\xi^{(J)}_G(\mathbf{Q}) = \frac{4\pi}{V} \cdot v'(\mathbf{G}) \cdot \left[\sum_\mathbf{k} \sum_i |f^{(J)}_i(\mathbf{G}, \mathbf{Q}, \mathbf{k})|\right]^2$$

where $v'(\mathbf{G}) = 1/|\mathbf{G}|^2$ is the regularised Coulomb kernel. The result has shape $(N_k, N_\text{pw})$ for each spin channel $J$.

In [6]:
xi = calc.compute_xi()

print("ξ contributions:")
for J in (0, 1):
    print(f"  J={J}: shape {xi[J].shape}, "
          f"sum(ξ) = {np.sum(xi[J]):.4f}, "
          f"max(ξ) = {np.max(xi[J]):.4f}")

# The soft two-body contribution to λ is (1/4) * sum over J,Q,G of ξ
soft_contribution = 0.25 * sum(np.sum(xi[J]) for J in (0, 1))
print(f"\nSoft two-body contribution to λ: {soft_contribution:.4f}")

ξ contributions:
  J=0: shape (8, 511), sum(ξ) = 38.5359, max(ξ) = 0.6545
  J=1: shape (8, 511), sum(ξ) = 38.5359, max(ξ) = 0.6545

Soft two-body contribution to λ: 19.2680


## Step 5: PAW Contribution $\chi$ --- `compute_chi()`

Accumulates the PAW augmentation contribution:

$$\chi^{a,J}_{rs}(\mathbf{Q}) = \left[\sum_\mathbf{k} \sum_i |f^{a,J}_{i,rs}(\mathbf{Q},\mathbf{k})|\right]^2$$

The result is laid out on dense $(n_a \times n_a)$ matrices for each (atom, $J$, $\mathbf{Q}$) triple.

In [7]:
chi = calc.compute_chi()

print("χ contributions:")
for (a, J), ch in sorted(chi.items()):
    print(f"  (atom={a}, J={J}): shape {ch.shape}, "
          f"sum(χ) = {np.sum(ch):.4f}, max(χ) = {np.max(ch):.4f}")

χ contributions:
  (atom=0, J=0): shape (8, 5, 5), sum(χ) = 0.0000, max(χ) = 0.0000
  (atom=0, J=1): shape (8, 5, 5), sum(χ) = 0.0000, max(χ) = 0.0000


## Step 6: One-Body Eigenvalues --- `diagonalize_one_plus_two_body()`

Diagonalises the effective one-body matrix $h'(\mathbf{k})$ to get eigenvalues $\varepsilon_i(\mathbf{k})$. The effective one-body matrix includes a mean-field exchange correction:

$$h'(\mathbf{k})_{ij} = h(\mathbf{k})_{ij} - \frac{1}{2} \sum_{\mathbf{k}',l} \left[\kappa_{\mathbf{k}i,\mathbf{k}'l,\mathbf{k}j,\mathbf{k}'l} - 2\,\kappa_{\mathbf{k}i,\mathbf{k}j,\mathbf{k}'l,\mathbf{k}'l}\right]$$

Three paths are possible depending on available data:
1. **Both $h_{pq}$ and $\kappa_{pqrs}$** --- full mean-field correction $h'(\mathbf{k}) = h(\mathbf{k}) - \tfrac{1}{2}\,\text{shift}(\mathbf{k})$
2. **Only $h_{pq}$** (our case) --- diagonalise $h(\mathbf{k},\mathbf{k})$ directly (the "fast path")
3. **Neither** --- return zeros (two-body only mode)

### Handling the $\kappa$ skip

Our test data was exported with `write_two_body=False`, so `kappa_pqrs` is `None`. In this case, `diagonalize_one_plus_two_body()` takes path 2: it diagonalises the raw $h(\mathbf{k})$ matrix directly, without subtracting the mean-field exchange shift. For the LCBO test system, this changes $\lambda$ by only ~0.02% --- well below other sources of error.

In [8]:
eps_one = calc.diagonalize_one_plus_two_body()

print(f"One-body eigenvalues: shape {eps_one.shape}  (Nk, Nb)")
print(f"  = eigenvalues ε_i(k) for each k-point and band\n")

print("Eigenvalues by k-point:")
for k in range(calc.Nk):
    vals = eps_one[k, :]
    formatted = ", ".join(f"{v:+.4f}" for v in vals)
    print(f"  k={k}: [{formatted}]")

one_body_contribution = np.sum(np.abs(eps_one))
print(f"\nOne-body contribution to λ: Σ|ε_i(k)| = {one_body_contribution:.4f}")

One-body eigenvalues: shape (8, 5)  (Nk, Nb)
  = eigenvalues ε_i(k) for each k-point and band

Eigenvalues by k-point:
  k=0: [-1.0804, +1.9344, +3.5872, +3.5958, +3.5958]
  k=1: [-0.4645, +2.1169, +2.2491, +3.3705, +3.3779]
  k=2: [-0.4645, +2.1169, +2.2491, +3.3705, +3.3779]
  k=3: [-0.2565, +1.8646, +2.1945, +3.0024, +3.0077]
  k=4: [-0.4645, +2.1169, +2.2491, +3.3705, +3.3779]
  k=5: [-0.2565, +1.8646, +2.1945, +3.0024, +3.0077]
  k=6: [-0.2565, +1.8646, +2.1945, +3.0024, +3.0077]
  k=7: [-0.4644, +2.1107, +2.2472, +3.3779, +3.3779]

One-body contribution to λ: Σ|ε_i(k)| = 91.0858


## Step 7: Assemble $\lambda$ --- `lambda_one_norm()`

Finally, the three contributions are assembled into the total one-norm:

$$\lambda = \underbrace{\sum_{k,i} |\varepsilon_i(\mathbf{k})|}_{\text{one-body}} + \underbrace{\frac{1}{4} \sum_J \sum_\mathbf{Q} \sum_\mathbf{G} \xi^{(J)}_\mathbf{G}(\mathbf{Q})}_{\text{soft two-body}} + \underbrace{\frac{1}{4} \sum_J \sum_\mathbf{Q} \sum_a \sum_{r,s} |\varepsilon^a_{rs}|\,\chi^{a,J}_{rs}(\mathbf{Q})}_{\text{PAW two-body}}$$

Since we already ran all the individual steps, `lambda_one_norm()` uses the cached results.

In [9]:
lam = calc.lambda_one_norm()

print(f"One-norm \u03bb = {lam:.4f}")
print(f"\nContribution breakdown:")
print(f"  One-body (\u03a3|\u03b5_i(k)|):        {one_body_contribution:10.4f}  ({one_body_contribution/lam:5.1%})")
print(f"  Soft two-body (\u00bc\u00b7\u03a3\u03be):        {soft_contribution:10.4f}  ({soft_contribution/lam:5.1%})")
paw_contribution = lam - one_body_contribution - soft_contribution
print(f"  PAW two-body (\u00bc\u00b7\u03a3|\u03b5|\u00b7\u03c7):     {paw_contribution:10.4f}  ({paw_contribution/lam:5.1%})")
print(f"  Total \u03bb:                     {lam:10.4f}")
print(f"\nThe query complexity of QPE scales as O(\u03bb / \u03b5_QPE).")

One-norm λ = 110.3537

Contribution breakdown:
  One-body (Σ|ε_i(k)|):           91.0858  (82.5%)
  Soft two-body (¼·Σξ):           19.2680  (17.5%)
  PAW two-body (¼·Σ|ε|·χ):         0.0000  ( 0.0%)
  Total λ:                       110.3537

The query complexity of QPE scales as O(λ / ε_QPE).


## Average Rank

The average rank $R^{(\ell \neq 0)}$ counts how many non-zero singular values appear across all two-body blocks, normalised by the total label count $L = 2 N_k M$. This determines the QROAM table size and thus the Toffoli scaling.

$R^{(0)}$ is the number of non-zero one-body eigenvalues --- the effective "one-body rank".

In [10]:
R_avg, R0 = calc.compute_average_rank()

print(f"Average two-body rank R_avg = {R_avg:.2f}")
print(f"One-body rank R0 = {R0:.0f}")
print(f"\nThese feed into the ResourceEstimator as Rl and R0.")

Average two-body rank R_avg = 56.11
One-body rank R0 = 40

These feed into the ResourceEstimator as Rl and R0.


## Summary

We walked through the seven-step pipeline that computes the Hamiltonian one-norm $\lambda$:

1. **`compute_soft_modes()`** --- FFT + batched SVD of the pair density
2. **`diagonalize_Ca()`** --- Eigendecompose on-site $C^a$ tensors
3. **`compute_paw_modes()`** --- Contract $D^a$ with $C$ eigenvectors + batched SVD
4. **`compute_xi()`** --- Accumulate soft contribution $\xi$
5. **`compute_chi()`** --- Accumulate PAW contribution $\chi$
6. **`diagonalize_one_plus_two_body()`** --- One-body eigenvalues $\varepsilon_i(\mathbf{k})$
7. **`lambda_one_norm()`** --- Assemble $\lambda$

Next: **05_resource_estimation.ipynb** shows how to convert $\lambda$ and the rank statistics into Toffoli gate and qubit counts.